# Job Salary Estimator — DAY 3: Evaluation, Baselines, Traditional ML

Evaluate predictors, set baselines, and train traditional ML models.

In [ ]:
import random
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.ensemble import RandomForestRegressor
from job_salary.evaluator import evaluate
from job_salary.items import Job

In [ ]:
# Load data - run Day 1 & 2 first, or use from_hub
try:
    train, val, test = Job.from_hub('ed-donner/job_salaries_processed')  # or your hub path
except Exception:
    from datasets import load_dataset
    from job_salary.parser import parse
    from tqdm.notebook import tqdm
    ds = load_dataset('lukebarousse/data_jobs', split='train', trust_remote_code=True)
    items = [parse(dp) for dp in tqdm(ds.select(range(10000)))]
    items = [j for j in items if j is not None]
    n = len(items)
    train, val, test = items[:int(0.8*n)], items[int(0.8*n):int(0.9*n)], items[int(0.9*n):]

print(f'Train: {len(train):,}, Val: {len(val):,}, Test: {len(test):,}')

In [ ]:
def get_text(job):
    return job.summary if job.summary else job.full or ''

In [ ]:
# Baselines
def random_pricer(job):
    return random.randint(30_000, 300_000)

train_avg = sum(j.salary for j in train) / len(train)
def constant_pricer(job):
    return train_avg

random.seed(42)
evaluate(random_pricer, test, size=min(200, len(test)))
evaluate(constant_pricer, test, size=min(200, len(test)))

In [ ]:
# Traditional ML - Linear Regression on text length & simple features
def get_features(job):
    text = get_text(job)
    return {
        'text_length': len(text),
        'remote': 1 if job.remote else 0,
        'has_remote': 1 if 'remote' in (text or '').lower() else 0
    }

train_df = pd.DataFrame([get_features(j) for j in train])
train_df['salary'] = [j.salary for j in train]
test_df = pd.DataFrame([get_features(j) for j in test])

X_train = train_df[['text_length', 'remote', 'has_remote']]
y_train = train_df['salary']
X_test = test_df[['text_length', 'remote', 'has_remote']]

lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred = lr.predict(X_test)
print(f'MSE: {mean_squared_error([j.salary for j in test[:len(y_pred)]], y_pred):,.0f}')
print(f'R2: {r2_score([j.salary for j in test[:len(y_pred)]], y_pred):.2%}')

def lr_pricer(job):
    X = pd.DataFrame([get_features(job)])
    return max(0, lr.predict(X)[0])

In [ ]:
evaluate(lr_pricer, test, size=min(200, len(test)))

In [ ]:
# Bag-of-words + Linear Regression
docs = [get_text(j) for j in train]
prices = np.array([float(j.salary) for j in train])

vec = CountVectorizer(max_features=2000, stop_words='english')
X = vec.fit_transform(docs)

reg = LinearRegression()
reg.fit(X, prices)

def bow_lr_pricer(job):
    x = vec.transform([get_text(job)])
    return max(0, reg.predict(x)[0])

evaluate(bow_lr_pricer, test, size=min(200, len(test)))

In [ ]:
# Random Forest
subset = min(15_000, len(train))
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X[:subset], prices[:subset])

def rf_pricer(job):
    x = vec.transform([get_text(job)])
    return max(0, rf.predict(x)[0])

evaluate(rf_pricer, test, size=min(200, len(test)))

In [ ]:
# XGBoost (if installed)
try:
    import xgboost as xgb
    xgb_model = xgb.XGBRegressor(n_estimators=100, random_state=42, n_jobs=-1)
    xgb_model.fit(X, prices)
    def xgb_pricer(job):
        x = vec.transform([get_text(job)])
        return max(0, xgb_model.predict(x)[0])
    evaluate(xgb_pricer, test, size=min(200, len(test)))
except ImportError:
    print('XGBoost not installed; skipping')